# PRISM analysis notebook

This notebook reproduces the final evaluation analysis for PRISM using only local files already included with the project. It does **not** rerun scraping, does **not** call live LLM APIs, and does **not** launch the Streamlit app.

PRISM studies how multilingual Wikipedia and related language-specific sources can answer the same question differently. The evaluation asks whether an LLM classifier can assign cross-language disagreement labels consistently against human gold labels.


## 1. Project overview

PRISM's research aim is to make cross-language disagreement visible. A single search query can look settled in one language internet while another language edition foregrounds a different date, inventor, cause, outcome, or omission. For digital humanities and information science, this matters because language communities do not only translate facts, they often organize knowledge through different historical memories, attribution norms, and definitions.

This notebook reproduces two final evaluation tracks:

- **Eval 1: Sacred / CaseBook12**: 12 curated Case Book examples with cleaner, manually assembled multilingual evidence.
- **Eval 2: Weak15**: 15 matched weak-set cases from a noisier real-world candidate pipeline. The original adjudicated weak set had 17 cases, but 2 were filtered before classification because multilingual preprocessing had insufficient or missing lead content.


## 2. Data and taxonomy

This notebook summarizes the main outputs of the PRISM dataset and evaluation pipeline. The full scraping and classifier code is stored in the `scripts/` folder. This notebook does not rerun the full LLM classifier because that step is slow and API-dependent. Instead, it loads the generated CSV and JSON outputs from `weak_set/` and `sacred_set/` and displays the main results.


In [41]:
from pathlib import Path
import pandas as pd
import json

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

required_files = [
    "weak_set/kept_cases.csv",
    "weak_set/drop_log.csv",
    "weak_set/drop_summary.json",
    "weak_set/predictions.csv",
    "sacred_set/predictions_casebook12.csv",
    "scripts/wiki_scraper.py",
    "scripts/run_weak_classifier.py",
    "scripts/run_casebook_classifier.py",
    "README.md",
    "requirements.txt",
]

inventory = pd.DataFrame({
    "file": required_files,
    "exists": [Path(f).exists() for f in required_files],
})

inventory

,file,exists
0,weak_set/kept_cases.csv,True
1,weak_set/drop_log.csv,True
2,weak_set/drop_summary.json,True
3,weak_set/predictions.csv,True
4,sacred_set/predictions_casebook12.csv,True
5,scripts/wiki_scraper.py,True
6,scripts/run_weak_classifier.py,True
7,scripts/run_casebook_classifier.py,True
8,README.md,True
9,requirements.txt,True


In [42]:
missing = inventory.loc[~inventory["exists"], "file"].tolist()

print("Missing files:", missing)

assert not missing, f"Missing required local files: {missing}"

print("All required local files are present.")

Missing files: []
All required local files are present.


In [43]:
taxonomy = pd.DataFrame([
    {'Label': 'A', 'Type': 'Factual divergence', 'Plain-language meaning': 'Sources assert different concrete facts, such as dates, numbers, names, or places.'},
    {'Label': 'B', 'Type': 'Attribution', 'Plain-language meaning': 'Sources credit different people, groups, or countries with an invention, discovery, or action.'},
    {'Label': 'C', 'Type': 'Outcome', 'Plain-language meaning': 'Sources disagree on who won, lost, or whether an event had a settled outcome.'},
    {'Label': 'D', 'Type': 'Framing', 'Plain-language meaning': 'Sources tell the same story with different causes, moral emphasis, or narrative center.'},
    {'Label': 'E', 'Type': 'Definitional boundary', 'Plain-language meaning': 'Sources draw the category boundary differently, such as when an event starts or ends.'},
    {'Label': 'F', 'Type': 'Omission', 'Plain-language meaning': 'One source leaves out information that another treats as central to the answer.'},
])
print(taxonomy.to_string(index=False))


Label                  Type                                                                         Plain-language meaning
    A    Factual divergence             Sources assert different concrete facts, such as dates, numbers, names, or places.
    B           Attribution Sources credit different people, groups, or countries with an invention, discovery, or action.
    C               Outcome                  Sources disagree on who won, lost, or whether an event had a settled outcome.
    D               Framing        Sources tell the same story with different causes, moral emphasis, or narrative center.
    E Definitional boundary          Sources draw the category boundary differently, such as when an event starts or ends.
    F              Omission                One source leaves out information that another treats as central to the answer.


In [44]:
# Load current project outputs

kept_cases = pd.read_csv("weak_set/kept_cases.csv")
drop_log = pd.read_csv("weak_set/drop_log.csv")
weak_predictions = pd.read_csv("weak_set/predictions.csv")
casebook_predictions = pd.read_csv("sacred_set/predictions_casebook12.csv")

loaded_inputs = pd.DataFrame([
    {
        "Dataset": "Weak-set kept cases",
        "Rows": len(kept_cases),
        "Columns": ", ".join(kept_cases.columns),
    },
    {
        "Dataset": "Weak-set dropped cases",
        "Rows": len(drop_log),
        "Columns": ", ".join(drop_log.columns),
    },
    {
        "Dataset": "Weak-set predictions",
        "Rows": len(weak_predictions),
        "Columns": ", ".join(weak_predictions.columns),
    },
    {
        "Dataset": "Case Book predictions",
        "Rows": len(casebook_predictions),
        "Columns": ", ".join(casebook_predictions.columns),
    },
])

loaded_inputs

,Dataset,Rows,Columns
0,Weak-set kept cases,133,"id, entity, question, keep, drop_reasons, en_status, en_lead_sentences, en_url, zh_status, zh_lead_sentences, zh_url..."
1,Weak-set dropped cases,17,"id, entity, question, keep, drop_reasons, en_status, en_lead_sentences, en_url, zh_status, zh_lead_sentences, zh_url..."
2,Weak-set predictions,133,"id, entity, question, predicted_labels, summary, omission_notes, confidence, raw_prediction"
3,Case Book predictions,12,"id, case_num, title, question, predicted_labels, summary, omission_notes, confidence, raw_prediction"


## 3. Evaluation method

This notebook summarizes two prediction outputs produced by the PRISM pipeline.

The Case Book output contains classifier predictions for the 12 hand-written Case Book cases. These rows use IDs from `Case-1` to `Case-12`, so they can be matched with the Case Book gold labels in a separate evaluation step.

The weak-set output contains classifier predictions for the 133 weak-set cases that survived scraping and filtering. These rows use candidate IDs such as `T1-03` and `T2-05`.

The notebook does not rerun the full LLM classifier because that step is slow and API-dependent. Instead, it loads the existing CSV outputs and reports their row counts, ID systems, and predicted labels.


In [45]:
# Evaluation method summary

evaluation_method = pd.DataFrame([
    {
        "Evaluation part": "Case Book evaluation",
        "Input file": "sacred_set/predictions_casebook12.csv",
        "Number of prediction rows": len(casebook_predictions),
        "ID system": "Case-1 to Case-12",
        "Purpose": "Compare classifier predictions with the 12 hand-written Case Book cases."
    },
    {
        "Evaluation part": "Weak-set evaluation",
        "Input file": "weak_set/predictions.csv",
        "Number of prediction rows": len(weak_predictions),
        "ID system": "Candidate IDs such as T1-03",
        "Purpose": "Evaluate the classifier on the weak-set cases that survived scraping and filtering."
    }
])

evaluation_method


,Evaluation part,Input file,Number of prediction rows,ID system,Purpose
0,Case Book evaluation,sacred_set/predictions_casebook12.csv,12,Case-1 to Case-12,Compare classifier predictions with the 12 hand-written Case Book cases.
1,Weak-set evaluation,weak_set/predictions.csv,133,Candidate IDs such as T1-03,Evaluate the classifier on the weak-set cases that survived scraping and filtering.


## 4. Case Book prediction output

This section displays classifier predictions for the 12 curated Case Book examples. These cases are a cleaner benchmark for cross-language disagreement types because the multilingual evidence is more controlled than in the weak set.

The prediction file uses IDs from `Case-1` to `Case-12`, so it can be matched with the Case Book gold labels in a separate evaluation step.


In [46]:
# Case Book prediction output

casebook_predictions = pd.read_csv("sacred_set/predictions_casebook12.csv")

print("Case Book prediction rows:", len(casebook_predictions))
print("Case Book IDs:", casebook_predictions["id"].tolist())

casebook_predictions[[
    "id",
    "case_num",
    "title",
    "question",
    "predicted_labels",
    "confidence"
]]

Case Book prediction rows: 12
Case Book IDs: ['Case-1', 'Case-2', 'Case-3', 'Case-4', 'Case-5', 'Case-6', 'Case-7', 'Case-8', 'Case-9', 'Case-10', 'Case-11', 'Case-12']


,id,case_num,title,question,predicted_labels,confidence
0,Case-1,1,Three Start Dates for One World War,Three Start Dates for One World War,Type E — Definitional Boundary; Type C — Outcome,high
1,Case-2,2,Who Invented Printing?,“Who invented the printing press?” / “ 谁 发 明 了 活 字 印 刷？ ” / « Кто изобрёл книгопечатание ?» Sources Compared English...,Type B — Attribution; Type F — Omission,high
2,Case-3,3,Three Winners of the War of 1812,Three Winners of the War of 1812,Type C — Outcome; Type D — Framing,high
3,Case-4,4,What Were the Opium Wars About?,"“What caused the Opium Wars?” / “ 鸦 片 战 争 的 起 因 是 什 么？ ” Sources Compared English Wikipedia, Chinese Wikipedia, and ...",Type D — Framing,high
4,Case-5,5,When Did the Roman Empire Fall?,“When did the Roman Empire fall?” / “ 罗 马 帝 国 何 时 灭 亡？ ” / « Πότε έπεσε η Ρω μ αϊκή Αυτοκρατορία ;» Sources Compared...,Type E — Definitional Boundary; Type D — Framing,high
5,Case-6,6,Who Discovered America?,Who Discovered America?,Type B — Attribution; Type D — Framing; Type F — Omission,high
6,Case-7,7,Naming and Memory of the Korean War,Naming and Memory of the Korean War,Type D — Framing; Type C — Outcome,medium
7,Case-8,8,When the Cold War Ended,When the Cold War Ended,Type E — Definitional Boundary; Type A — Factual Divergence,high
8,Case-9,9,Who Invented the Telephone?,Who Invented the Telephone?,Type B — Attribution; Type F — Omission,high
9,Case-10,10,Who Discovered Penicillin?,Who Discovered Penicillin?,Type B — Attribution; Type F — Omission,high


**Interpretation.** This section shows the classifier predictions for the 12 Case Book cases. These cases are more controlled than the weak set because they were curated by hand. The prediction file uses `Case-1` to `Case-12` IDs, so it can be matched with gold labels in a separate evaluation step. This notebook displays the prediction output but does not recompute evaluation metrics.

## 5. Weak-set prediction output

This section displays the classifier predictions for the weak-set cases.

The weak set started from 150 candidate cases. After scraping and filtering, 133 cases were kept and passed to the classifier. These prediction rows use candidate IDs such as `T1-03` and `T2-05`.

This output is separate from the Case Book output because the weak set uses a different ID system and comes from a broader, noisier candidate pipeline.

Some weak-set candidates were dropped before classification because of missing language pages or very short lead text. The dropped cases and reasons are recorded in `weak_set/drop_log.csv`.

In [47]:
# Weak-set prediction output

weak_predictions = pd.read_csv("weak_set/predictions.csv")

print("Weak-set prediction rows:", len(weak_predictions))
print("Weak-set prediction columns:")
print(weak_predictions.columns.tolist())

weak_predictions[[
    "id",
    "entity",
    "question",
    "predicted_labels",
    "confidence"
]].head(20)

Weak-set prediction rows: 133
Weak-set prediction columns:
['id', 'entity', 'question', 'predicted_labels', 'summary', 'omission_notes', 'confidence', 'raw_prediction']


,id,entity,question,predicted_labels,confidence
0,T1-03,Six-Day War,Who won the Six-Day War?,Type C — Outcome; Type F — Omission,medium
1,T1-04,Yom Kippur War,Who won the Yom Kippur War?,Type C — Outcome; Type F — Omission,medium
2,T1-05,Sino-Indian War of 1962,Who won the Sino-Indian War of 1962?,Type C — Outcome,medium
3,T1-06,Iran-Iraq War,Who won the Iran-Iraq War?,Type C — Outcome; Type D — Framing,medium
4,T1-07,First Anglo-Afghan War,Who won the First Anglo-Afghan War?,Type C — Outcome; Type F — Omission,high
5,T1-08,Second Boer War,Who won the Second Boer War?,Type F — Omission,high
6,T1-09,Boxer Rebellion,Who won the Boxer Rebellion?,Type F — Omission,high
7,T1-10,Russo-Japanese War,Who won the Russo-Japanese War?,Type C — Outcome; Type F — Omission,high
8,T1-11,Russo-Turkish War of 1877-78,Who won the Russo-Turkish War of 1877-78?,Type C — Outcome; Type F — Omission,high
9,T1-12,Chaco War,Who won the Chaco War?,ERROR,error


In [48]:
# Weak-set predicted label distribution

weak_predictions["predicted_labels"].value_counts().head(20)

predicted_labels
Type F — Omission                                                          33
No label returned                                                          14
Type D — Framing                                                           11
Type B — Attribution; Type F — Omission                                    10
Type D — Framing; Type F — Omission                                        10
Type E — Definitional Boundary                                              8
Type C — Outcome; Type F — Omission                                         7
Type C — Outcome                                                            6
Type C — Outcome; Type D — Framing                                          3
Type B — Attribution                                                        3
Type B — Attribution; Type E — Definitional Boundary                        3
Type B — Attribution; Type D — Framing                                      3
Type A — Factual Divergence; Type F — Omission 

In [49]:
# Weak-set scraping and filtering summary

kept_cases = pd.read_csv("weak_set/kept_cases.csv")
drop_log = pd.read_csv("weak_set/drop_log.csv")

summary = pd.DataFrame({
    "Stage": [
        "Initial weak-set candidates",
        "Kept after scraping and filtering",
        "Dropped before classification",
        "Weak-set predictions"
    ],
    "Count": [
        150,
        len(kept_cases),
        len(drop_log),
        len(weak_predictions)
    ]
})

summary

,Stage,Count
0,Initial weak-set candidates,150
1,Kept after scraping and filtering,133
2,Dropped before classification,17
3,Weak-set predictions,133


In [50]:
# Human-evaluated weak subset note

selected_subset_note = pd.DataFrame({
    "Item": [
        "Human-evaluated weak-set IDs selected",
        "Matched prediction rows",
        "Missing prediction rows",
        "Missing IDs"
    ],
    "Value": [
        17,
        15,
        2,
        "T1-01, T2-15"
    ]
})

selected_subset_note

,Item,Value
0,Human-evaluated weak-set IDs selected,17
1,Matched prediction rows,15
2,Missing prediction rows,2
3,Missing IDs,"T1-01, T2-15"


**Interpretation.** This section summarizes the weak-set prediction output. The full weak-set file contains 133 predictions after scraping and filtering. A later human-evaluated subset selected 17 IDs, but only 15 matched the prediction file because `T1-01` and `T2-15` had already been dropped before classification.

## 6. Case Book vs Weak Set comparison

This section compares the two main output tracks in the project.

The Case Book output is smaller and more controlled. It contains 12 curated cases with `Case-1` to `Case-12` IDs.

The weak-set output is larger and noisier. It started from 150 candidate cases, kept 133 after scraping and filtering, and produced 133 classifier prediction rows.

This comparison does not report precision, recall, F1, or a confusion matrix. It summarizes the scale and role of the two outputs.

In [51]:
# Case Book vs Weak Set comparison

comparison = pd.DataFrame([
    {
        "Output track": "Case Book",
        "Input size": 12,
        "Prediction rows": len(casebook_predictions),
        "ID system": "Case-1 to Case-12",
        "Role": "Controlled benchmark cases"
    },
    {
        "Output track": "Weak set",
        "Input size": 150,
        "Prediction rows": len(weak_predictions),
        "ID system": "Candidate IDs such as T1-03",
        "Role": "Broader candidate set after scraping and filtering"
    }
])

comparison

,Output track,Input size,Prediction rows,ID system,Role
0,Case Book,12,12,Case-1 to Case-12,Controlled benchmark cases
1,Weak set,150,133,Candidate IDs such as T1-03,Broader candidate set after scraping and filtering


## 7. Limitations

Several limits shape how these outputs should be interpreted:

- This notebook reports prediction outputs and dataset summaries. It does not recompute precision, recall, F1, or a confusion matrix.
- The Case Book output is small, with 12 curated cases. It is useful as a controlled benchmark, but it is not large enough to represent all possible cross-language disagreements.
- The weak-set output is larger, with 133 prediction rows, but it is noisier because the cases were generated from templates and collected through an automated scraping pipeline.
- The weak-set pipeline depends on Wikipedia language availability. Some candidates were dropped because one or more language pages were missing or the lead text was too short.
- The later human-evaluated weak subset selected 17 IDs, but only 15 had matching prediction rows because `T1-01` and `T2-15` had already been dropped before classification.
- Search results, language links, text extraction, and LLM classification can all introduce noise.
- Multi-label classification is difficult because one case may contain more than one disagreement type.

## 8. Conclusion

This notebook summarizes the main outputs of the PRISM cross-language search prototype.

The project produced two separate prediction outputs. The first is the Case Book output, which contains predictions for 12 curated cases with `Case-1` to `Case-12` IDs. The second is the weak-set output, which started from 150 candidate cases, kept 133 after scraping and filtering, and produced 133 classifier prediction rows with candidate IDs such as `T1-03`.

The main result is that PRISM can organize multilingual evidence side by side and produce disagreement labels for both controlled Case Book cases and a larger weak-set pipeline. At the same time, the weak-set process shows that retrieval quality and language-page availability strongly affect what can be classified.

Future work should improve retrieval quality, expand the human-evaluated subset, and evaluate the classifier outputs against gold labels more systematically.

In [52]:
# Final notebook check

final_check = pd.DataFrame([
    {"Check": "Notebook loads local output files", "Status": "Passed"},
    {"Check": "No API keys are required to view this notebook output", "Status": "Passed"},
    {"Check": "No live scraping rerun is required to view this notebook output", "Status": "Passed"},
    {"Check": "Weak-set prediction file is loaded", "Status": f"Passed, {len(weak_predictions)} rows"},
    {"Check": "Case Book prediction file is loaded", "Status": f"Passed, {len(casebook_predictions)} rows"},
])

final_check

,Check,Status
0,Notebook loads local output files,Passed
1,No API keys are required to view this notebook output,Passed
2,No live scraping rerun is required to view this notebook output,Passed
3,Weak-set prediction file is loaded,"Passed, 133 rows"
4,Case Book prediction file is loaded,"Passed, 12 rows"
